In [15]:
import pandas as pd
import numpy as np

In [16]:
train_panel_in = "../../../data/final/train_panel.parquet"
bike_panel_in = "../../../data/final/bike_panel.parquet"
uber_panel_in = "../../../data/final/uber_panel.parquet"

In [17]:
panel = pd.concat([pd.read_parquet(train_panel_in),
                   pd.read_parquet(bike_panel_in),
                   pd.read_parquet(uber_panel_in)],
                  keys=['train','bike','uber'], names=['transit']).reset_index('transit')

## Contextualized Effect

In [22]:
n_trt = panel[(panel.UCMP==1)&(panel.DNC==1)].groupby('transit').size()
n_ctl = panel[(panel.UCMP==0)&(panel.DNC==1)].groupby('transit').size()
mean_trt = panel[(panel.UCMP==1)&(panel.DNC==0)].groupby(['transit','id'])['rides'].mean().groupby('transit').mean()
mean_ctl = panel[(panel.UCMP==0)&(panel.DNC==0)].groupby(['transit','id'])['rides'].mean().groupby('transit').mean()
# The geom mean vastly under-estimates the scale of the changes. because the total changes
# are in fact driven by the high-ridership tail of the distribution, which is discounted in the geom mean.
geo_mean_trt = panel[(panel.UCMP==1)&(panel.DNC==0)].groupby(['transit','id'])['logrides'].mean().groupby('transit').mean()
geo_mean_ctl = panel[(panel.UCMP==0)&(panel.DNC==0)].groupby(['transit','id'])['logrides'].mean().groupby('transit').mean()
exp_mean_trt = np.exp(geo_mean_trt)
exp_mean_ctl = np.exp(geo_mean_ctl)
pct_trt = (np.exp(beta3) - 1)
pct_ctl = (np.exp(beta1) - 1)
pct_trt.index = pct_trt.index.map(lambda x: x.replace('DiD (','').replace(')','').lower())
pct_ctl.index = pct_ctl.index.map(lambda x: x.replace('DiD (','').replace(')','').lower())
delta_trt = pct_trt * mean_trt * n_trt
delta_ctl = pct_ctl * mean_ctl * n_ctl
tmp = pd.DataFrame({'trt':delta_trt, 'ctl':delta_ctl}).round(0)
delta_trt = pct_trt * geo_mean_trt * n_trt
delta_ctl = pct_ctl * geo_mean_ctl * n_ctl
tmp2 = pd.DataFrame({'geotrt':delta_trt, 'geoctl':delta_ctl}).round(0)
pd.concat([tmp,tmp2],axis=1)

,trt,ctl,geotrt,geoctl
bike,2987.0,-2538.0,220.0,-196.0
train,62426.0,-142951.0,287.0,-397.0
uber,24047.0,-42733.0,178.0,-825.0
